In [1]:
# CELL 23 — LightGBM on the full dataset: train on train docs, evaluate on dev docs (same protocol/tables as pilot CELL 11)
import os, glob, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

hits = sorted(glob.glob("/kaggle/input/**/full/full_features.csv", recursive=True))
assert hits, "full_features.csv not found - is BTP-full attached as input?"
IN_DIR = os.path.dirname(hits[0])
OUT_DIR = "/kaggle/working/results"
os.makedirs(OUT_DIR, exist_ok=True)
print("Input folder:", IN_DIR)

df = pd.read_csv(f"{IN_DIR}/full_features.csv")
fails = pd.read_csv(f"{IN_DIR}/full_failures.csv")
print("Rows:", len(df), "| documents:", df.orig_file.nunique(),
      "| train docs:", df[df.split == "train"].orig_file.nunique(), "| dev docs:", df[df.split == "dev"].orig_file.nunique())

# ---------- check 0: why documents failed, per family ----------
print("\n=== CHECK 0: FAILED DOCUMENTS PER FAMILY AND REASON ===")
fails["why"] = fails.stage + ": " + fails.reason.astype(str).str.slice(0, 40)
print(pd.crosstab(fails.why, fails.family).to_string())

META = ["has_producer", "producer_len", "moddate_eq_creation", "has_xmp", "tool_word_in_file"]
L1 = ["size_log", "version", "binary_comment", "n_eof", "n_startxref", "xref_table", "xref_stream", "obj_streams",
      "linearized", "true_revisions", "eol_crlf", "eol_cr", "eol_lf", "n_obj_defs", "obj_order_increasing",
      "obj_gen_nonzero", "obj_max_over_count", "obj_redefined", "obj_newline_dict", "dict_open_space",
      "dict_close_space", "stream_kw_crlf", "endobj_newline_before", "reals_dot_zero", "has_ID",
      "tool_word_in_file", "n_pages"]
L23 = ["cs_n", "cs_frac_flate", "cs_frac_lzw", "cs_frac_none", "cs_n_filters", "cs_frac_zlib_repro", "cs_n_hdr",
       "cs_n_zlevel", "cs_frac_sync", "cs_frac_adler", "cs_frac_junk", "cs_frac_len_indirect", "cs_ratio_std",
       "cs_ratio_max_z", "cs_n_sig", "cs_majority_share", "cs_n_singletons"]
L4 = ["upd_n_objs", "upd_touch_content", "upd_has_annot", "upd_has_page", "upd_has_sig"]
L5 = ["l5_n_annots", "l5_n_freetext", "l5_n_text_notes", "l5_n_img_xobj", "l5_n_fonts", "l5_n_std14_unembedded",
      "l5_frac_pages_multi_stream", "l5_max_streams_per_page"]
V1SET = sorted(set(META + L1 + L23 + L4))
FULL = sorted(set(V1SET + L5))
missing = [c for c in FULL if c not in df.columns]
assert not missing, f"Missing feature columns: {missing}"

GROUPS = {"META (metadata only)": META, "L1 (file style)": L1, "L2+L3 (compression + consistency)": L23,
          "L4 (last-update diff)": L4, "L5 (page resources)": L5, "L1-L4 (pilot v1 set)": V1SET, "FULL MODEL (L1-L5)": FULL}
PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)

def tpr_at(y, s, fpr):
    thr = np.quantile(s[y == 0], 1 - fpr)
    return float((s[y == 1] > thr).mean())

def auc_or_nan(y, s):
    try:
        return round(roc_auc_score(y, s), 3)
    except ValueError:
        return np.nan

def scores(y, s):
    y, s = np.asarray(y), np.asarray(s, dtype=float)
    return {"ROC-AUC": auc_or_nan(y, s), "PR-AUC": round(average_precision_score(y, s), 3),
            "Det@1%FA": round(tpr_at(y, s, 0.01), 3), "Det@5%FA": round(tpr_at(y, s, 0.05), 3)}

def fit(cols, train):
    return lgb.LGBMClassifier(**PARAMS).fit(train[cols], train["label"])

tr = df[df.split == "train"].reset_index(drop=True)
te = df[df.split == "dev"].reset_index(drop=True)
NO_EV = ["F2_digit_recompress", "F7_stamp_fake", "F8_textbox_fake", "G2_recompress", "G5_stamp_honest", "G6_textbox_honest"]
EVID = ~te.variant.isin(NO_EV)

# ---------- E1: main results ----------
rows, models = [], {}
rows.append({"Method": "Rule: %%EOF count > 1 (industry check)", **scores(te.label, te.n_eof)})
rows.append({"Method": "Rule: true revisions > 1", **scores(te.label, te.true_revisions)})
for g, cols in tqdm(GROUPS.items(), desc="E1 feature groups"):
    models[g] = fit(cols, tr)
    rows.append({"Method": f"LightGBM: {g}", **scores(te.label, models[g].predict_proba(te[cols])[:, 1])})
print("\n=== E1: MAIN RESULTS, all 15 versions (dev set) ===")
print(pd.DataFrame(rows).to_string(index=False))

te["score"] = models["FULL MODEL (L1-L5)"].predict_proba(te[FULL])[:, 1]
print("\n=== E1b: EVIDENCE-BEARING ONLY (excl. F2/F7/F8 and twins G2/G5/G6) ===")
print(pd.DataFrame([{"Method": "Rule: %%EOF count > 1", **scores(te.label[EVID], te.n_eof[EVID])},
                    {"Method": "LightGBM: FULL MODEL (L1-L5)", **scores(te.label[EVID], te.score[EVID])}]).to_string(index=False))

# ---------- threshold from TRAIN only (~5% false alarms, out-of-fold, grouped by document) ----------
oof = np.zeros(len(tr))
for a, b in tqdm(list(GroupKFold(n_splits=5).split(tr, tr.label, tr.orig_file)), desc="Threshold folds"):
    oof[b] = fit(FULL, tr.iloc[a]).predict_proba(tr.iloc[b][FULL])[:, 1]
THR = float(np.quantile(oof[tr.label.values == 0], 0.95))
te["flagged"] = (te.score > THR).astype(int)
print(f"\nThreshold (set on train docs for ~5% false alarms): {THR:.3f}")

# ---------- E2: per version ----------
print("\n=== E2: FLAG RATE PER VERSION (G = false alarms, F = detections) ===")
pv = te.groupby(["variant", "label"])["flagged"].mean().mul(100).round(1).reset_index()
pv.columns = ["variant", "label", "flagged_%"]
print(pv.to_string(index=False))

PAIRS = [("G1_resave", "F1_digit_resave"), ("G2_recompress", "F2_digit_recompress"), ("G3_note_incr", "F3_digit_incr"),
         ("G1_resave", "F4_value_resave"), ("G1_resave", "F5_cover_replace"), ("G4_linearize", "F6_digit_linearize"),
         ("G5_stamp_honest", "F7_stamp_fake"), ("G6_textbox_honest", "F8_textbox_fake")]
print("\n=== E2b: TWIN-PAIR AUC (forgery vs its honest twin | vs untouched original) ===")
for g, f in PAIRS:
    sub = te[te.variant.isin([g, f])]
    sub0 = te[te.variant.isin(["G0_original", f])]
    print(f"{f:20s} vs {g:18s}: {auc_or_nan(sub.label, sub.score):<6} | vs original: {auc_or_nan(sub0.label, sub0.score)}")

# ---------- E3: per tool ----------
print("\n=== E3: PER TOOL (AUC vs twin) ===")
for t in ["pikepdf", "pymupdf", "pypdf"]:
    out = []
    for g, f in [("G1_resave", "F1_digit_resave"), ("G1_resave", "F4_value_resave"), ("G2_recompress", "F2_digit_recompress")]:
        sub = te[(te.tool == t) & te.variant.isin([g, f])]
        out.append(f"{f.split('_')[0]}: {auc_or_nan(sub.label, sub.score)} (n={len(sub)})")
    print(f"{t:8s} | " + " | ".join(out))

# ---------- E4: per family ----------
print("\n=== E4: PER FAMILY (AUC) ===")
for fam in sorted(te.family.unique()):
    s_all, s_ev = te[te.family == fam], te[(te.family == fam) & EVID]
    print(f"{fam:28s} | dev docs: {s_all.orig_file.nunique():4d} | all versions: {auc_or_nan(s_all.label, s_all.score)}"
          f" | evidence-bearing: {auc_or_nan(s_ev.label, s_ev.score)}")

# ---------- E5: leave-one-tool-out ----------
print("\n=== E5: LEAVE-ONE-TOOL-OUT (train without the tool, test only on it) ===")
RW_TR = ["G0_original", "G1_resave", "G2_recompress", "F1_digit_resave", "F2_digit_recompress", "F4_value_resave"]
for t in tqdm(["pikepdf", "pymupdf", "pypdf"], desc="LOTO"):
    m = fit(FULL, tr[tr.variant.isin(RW_TR) & (tr.tool != t)])
    te_all = te[te.variant.isin(RW_TR[1:]) & (te.tool == t)]
    te_ev = te[te.variant.isin(["G1_resave", "F1_digit_resave", "F4_value_resave"]) & (te.tool == t)]
    print(f"unseen tool = {t:8s}: incl. laundering = {auc_or_nan(te_all.label, m.predict_proba(te_all[FULL])[:, 1])}"
          f" | resave forgeries only = {auc_or_nan(te_ev.label, m.predict_proba(te_ev[FULL])[:, 1])}")

# ---------- E7: rule-based localization (no training -> all documents) ----------
print("\n=== E7: RULE LOCALIZATION, all documents (odd-one-out stream -> edited page?) ===")
for v in ["F1_digit_resave", "F4_value_resave", "F5_cover_replace", "F6_digit_linearize", "F2_digit_recompress"]:
    sub = df[(df.variant == v) & df.edit_page.notna()]
    cov = sub[sub.loc_page >= 0]
    acc = round((cov.loc_page == cov.edit_page).mean() * 100, 1) if len(cov) else 0
    print(f"{v:20s}: fired in {round(len(cov) / max(len(sub), 1) * 100, 1)}% | correct page in {acc}% of those (n={len(sub)})")

# ---------- E8: feature importance + ablations ----------
imp = pd.Series(models["FULL MODEL (L1-L5)"].booster_.feature_importance("gain"), index=FULL)
print("\n=== E8: TOP 15 FEATURES (% of gain) ===")
print((imp / imp.sum() * 100).sort_values(ascending=False).head(15).round(1).to_string())
print("\n=== E8b: ABLATIONS (evidence-bearing dev rows, AUC) ===")
for name, cols in tqdm([("FULL", FULL), ("FULL without cs_n_hdr", [c for c in FULL if c != "cs_n_hdr"]),
                        ("FULL without L2+L3 (all compression features)", [c for c in FULL if c not in L23]),
                        ("FULL without L5", V1SET)], desc="Ablations"):
    m = fit(cols, tr)
    print(f"{name:48s}: {auc_or_nan(te.label[EVID], m.predict_proba(te[EVID][cols])[:, 1])}")

# ---------- 95% CIs: bootstrap over dev documents ----------
rng = np.random.default_rng(0)
docs = te.orig_file.unique()
by_doc = {d: te.index[te.orig_file == d].values for d in docs}
aa, ae = [], []
for _ in tqdm(range(300), desc="Bootstrap"):
    idx = np.concatenate([by_doc[d] for d in rng.choice(docs, size=len(docs), replace=True)])
    s = te.loc[idx]
    aa.append(roc_auc_score(s.label, s.score))
    se = s[~s.variant.isin(NO_EV)]
    ae.append(roc_auc_score(se.label, se.score))
print("\n=== 95% CONFIDENCE INTERVALS (FULL model, bootstrap over dev documents) ===")
print(f"All versions:          {np.mean(aa):.3f}  [{np.percentile(aa, 2.5):.3f}, {np.percentile(aa, 97.5):.3f}]")
print(f"Evidence-bearing only: {np.mean(ae):.3f}  [{np.percentile(ae, 2.5):.3f}, {np.percentile(ae, 97.5):.3f}]")

te.to_csv(f"{OUT_DIR}/lgbm_dev_predictions.csv", index=False)
print(f"\nSaved: {OUT_DIR}/lgbm_dev_predictions.csv")

Input folder: /kaggle/input/notebooks/atharvagandhi/btp-full/full
Rows: 40830 | documents: 2722 | train docs: 2319 | dev docs: 403

=== CHECK 0: FAILED DOCUMENTS PER FAMILY AND REASON ===
family                                              acrobat distiller for  acrobat distiller windows  acrobat pdfwriter for  adobe pdf library  corel pdf engine
why                                                                                                                                                             
generate: RuntimeError: file needed repair                              5                          0                      3                  0                 0
generate: RuntimeError: no editable number                             15                         45                      6                 50                 3
generate: RuntimeError: no numeric word for cover-                      0                          1                      0                  0                 0
generat

E1 feature groups:   0%|          | 0/7 [00:00<?, ?it/s]


=== E1: MAIN RESULTS, all 15 versions (dev set) ===
                                     Method  ROC-AUC  PR-AUC  Det@1%FA  Det@5%FA
     Rule: %%EOF count > 1 (industry check)    0.433   0.512     0.000     0.000
                   Rule: true revisions > 1    0.483   0.525     0.000     0.000
             LightGBM: META (metadata only)    0.550   0.566     0.008     0.058
                  LightGBM: L1 (file style)    0.736   0.767     0.160     0.255
LightGBM: L2+L3 (compression + consistency)    0.872   0.909     0.579     0.633
            LightGBM: L4 (last-update diff)    0.625   0.629     0.125     0.125
              LightGBM: L5 (page resources)    0.639   0.680     0.106     0.175
             LightGBM: L1-L4 (pilot v1 set)    0.901   0.923     0.603     0.643
               LightGBM: FULL MODEL (L1-L5)    0.908   0.926     0.604     0.645

=== E1b: EVIDENCE-BEARING ONLY (excl. F2/F7/F8 and twins G2/G5/G6) ===
                      Method  ROC-AUC  PR-AUC  Det@1%FA  Det@5%FA

Threshold folds:   0%|          | 0/5 [00:00<?, ?it/s]


Threshold (set on train docs for ~5% false alarms): 0.531

=== E2: FLAG RATE PER VERSION (G = false alarms, F = detections) ===
            variant  label  flagged_%
    F1_digit_resave      1       99.5
F2_digit_recompress      1        8.2
      F3_digit_incr      1      100.0
    F4_value_resave      1       99.5
   F5_cover_replace      1       99.8
 F6_digit_linearize      1       89.8
      F7_stamp_fake      1        6.5
    F8_textbox_fake      1        6.7
        G0_original      0        0.0
          G1_resave      0        5.0
      G2_recompress      0        7.9
       G3_note_incr      0        0.0
       G4_linearize      0        2.7
    G5_stamp_honest      0        6.5
  G6_textbox_honest      0        6.7

=== E2b: TWIN-PAIR AUC (forgery vs its honest twin | vs untouched original) ===
F1_digit_resave      vs G1_resave         : 0.996  | vs original: 1.0
F2_digit_recompress  vs G2_recompress     : 0.501  | vs original: 1.0
F3_digit_incr        vs G3_note_incr      

LOTO:   0%|          | 0/3 [00:00<?, ?it/s]

unseen tool = pikepdf : incl. laundering = 0.874 | resave forgeries only = 0.993
unseen tool = pymupdf : incl. laundering = 0.912 | resave forgeries only = 0.999
unseen tool = pypdf   : incl. laundering = 0.905 | resave forgeries only = 0.99

=== E7: RULE LOCALIZATION, all documents (odd-one-out stream -> edited page?) ===
F1_digit_resave     : fired in 82.8% | correct page in 99.5% of those (n=2722)
F4_value_resave     : fired in 82.8% | correct page in 99.5% of those (n=2722)
F5_cover_replace    : fired in 88.4% | correct page in 99.8% of those (n=2722)
F6_digit_linearize  : fired in 76.0% | correct page in 99.8% of those (n=2722)
F2_digit_recompress : fired in 8.2% | correct page in 19.4% of those (n=2722)

=== E8: TOP 15 FEATURES (% of gain) ===
cs_n_zlevel           41.9
cs_frac_zlib_repro    12.2
tool_word_in_file      8.8
l5_n_text_notes        6.1
cs_n_singletons        5.8
cs_n_hdr               5.5
stream_kw_crlf         3.1
cs_frac_none           2.2
eol_lf                 1

Ablations:   0%|          | 0/4 [00:00<?, ?it/s]

FULL                                            : 0.998
FULL without cs_n_hdr                           : 0.998
FULL without L2+L3 (all compression features)   : 0.874
FULL without L5                                 : 0.998


Bootstrap:   0%|          | 0/300 [00:00<?, ?it/s]


=== 95% CONFIDENCE INTERVALS (FULL model, bootstrap over dev documents) ===
All versions:          0.908  [0.905, 0.911]
Evidence-bearing only: 0.998  [0.997, 0.999]

Saved: /kaggle/working/results/lgbm_dev_predictions.csv


In [2]:
# CELL 24 — GNN setup on the full dataset: load graphs, consistency node features, normalization, splits (train / val-from-train / dev), model + training helpers
!pip install -q torch_geometric

import os, glob, gzip, pickle, time, zlib, random, warnings, gc
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, GINEConv
from torch_geometric.utils import scatter
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
torch.set_num_threads(4)

hg = sorted(glob.glob("/kaggle/input/**/full/full_graphs.pkl.gz", recursive=True))
hf = sorted(glob.glob("/kaggle/input/**/full/full_features.csv", recursive=True))
assert hg and hf, "full_graphs.pkl.gz / full_features.csv not found - is BTP-full attached as input?"
GRAPHS, FEATS = hg[0], hf[0]
RES_DIR = "/kaggle/working/gnn_full"
os.makedirs(RES_DIR, exist_ok=True)
print("Graphs:", GRAPHS)

t0 = time.time()
with gzip.open(GRAPHS, "rb") as fh:
    G = pickle.load(fh)
graphs, FEAT, NE = G["graphs"], list(G["feat_names"]), len(G["edge_keys"])
del G
feat_df = pd.read_csv(FEATS).set_index("out_file")
print(f"Loaded {len(graphs)} graphs in {time.time() - t0:.0f}s | base node features: {len(FEAT)} | edge types: {NE}")

# ---------- splits: train (minus 15% for validation) / val (from train docs) / dev (reporting only) ----------
info = pd.DataFrame([{k: g[k] for k in ["out_file", "orig_file", "family", "variant", "label", "tool", "twin",
                                         "edit_type", "edit_page", "split"]} for g in graphs])
info["is_val"] = (info.split == "train") & info.orig_file.map(lambda s: zlib.crc32((s + "|val").encode()) % 100 < 15)
tr_idx = info.index[(info.split == "train") & ~info.is_val].tolist()
va_idx = info.index[info.is_val].tolist()
te_idx = info.index[info.split == "dev"].tolist()
print(f"Graphs -> train: {len(tr_idx)} | val: {len(va_idx)} | dev: {len(te_idx)}")
print(f"Docs   -> train: {info.loc[tr_idx].orig_file.nunique()} | val: {info.loc[va_idx].orig_file.nunique()}"
      f" | dev: {info.loc[te_idx].orig_file.nunique()}")

# ---------- consistency node features (same as pilot CELL 14) ----------
C_CONTENT = FEAT.index("t_content")
FP = ["f_flate", "f_lzw", "f_dct", "f_none", "f_other", "h_789C", "h_78DA", "h_7801", "h_785E", "h_other",
      "zlevel", "zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio", "len_indirect", "entropy", "raw_len_log"]
SIG = ["f_flate", "f_lzw", "f_dct", "f_none", "f_other", "h_789C", "h_78DA", "h_7801", "h_785E", "h_other",
       "zlevel", "zlib_repro", "adler_ok", "sync_end", "tail_junk", "len_indirect"]
FP_I = [FEAT.index(c) for c in FP]
SIG_I = [FEAT.index(c) for c in SIG]
EXTRA = [f"dev_{c}" for c in FP] + ["odd_one_out", "is_singleton", "n_content_log"]

def consistency(X):
    n = X.shape[0]
    E = np.zeros((n, len(EXTRA)), dtype=np.float32)
    idx = np.where(X[:, C_CONTENT] > 0.5)[0]
    k = len(idx)
    E[:, -1] = np.log1p(k) / 5
    if k >= 2:
        S = X[idx][:, FP_I]
        tot = S.sum(0)
        E[idx, :len(FP)] = np.abs(S - (tot - S) / (k - 1))
        sig = np.round(X[idx][:, SIG_I], 3)
        _, inv, cnt = np.unique(sig, axis=0, return_inverse=True, return_counts=True)
        inv = np.asarray(inv).reshape(-1)
        E[idx, len(FP)] = 1 - cnt[inv] / k
        if k >= 3:
            E[idx, len(FP) + 1] = (cnt[inv] == 1).astype(np.float32)
    return E

ALL_NAMES = FEAT + EXTRA
BASE_COLS = list(range(len(FEAT)))
ALL_COLS = list(range(len(ALL_NAMES)))
tr_set = set(tr_idx)
XS = [None] * len(graphs)
S1 = np.zeros(len(ALL_NAMES)); S2 = np.zeros(len(ALL_NAMES)); NN = 0
for i, g in enumerate(tqdm(graphs, desc="Consistency features")):
    X = g["x"].astype(np.float32)
    if X.shape[0] == 0:
        X = np.zeros((1, len(FEAT)), dtype=np.float32)
    XA = np.concatenate([X, consistency(X)], 1)
    XS[i] = XA
    g["x"] = None
    if i in tr_set:
        S1 += XA.sum(0, dtype=np.float64)
        S2 += (XA.astype(np.float64) ** 2).sum(0)
        NN += XA.shape[0]
MU = (S1 / NN).astype(np.float32)
SD = np.sqrt(np.maximum(S2 / NN - (S1 / NN) ** 2, 0)).astype(np.float32)
SD[SD < 1e-6] = 1.0
print("Training nodes used for normalization:", NN)

# ---------- document-level features for CGNN_DOC (standardized on train graphs) ----------
META = ["has_producer", "producer_len", "moddate_eq_creation", "has_xmp", "tool_word_in_file"]
L1 = ["size_log", "version", "binary_comment", "n_eof", "n_startxref", "xref_table", "xref_stream", "obj_streams",
      "linearized", "true_revisions", "eol_crlf", "eol_cr", "eol_lf", "n_obj_defs", "obj_order_increasing",
      "obj_gen_nonzero", "obj_max_over_count", "obj_redefined", "obj_newline_dict", "dict_open_space",
      "dict_close_space", "stream_kw_crlf", "endobj_newline_before", "reals_dot_zero", "has_ID", "tool_word_in_file", "n_pages"]
L23 = ["cs_n", "cs_frac_flate", "cs_frac_lzw", "cs_frac_none", "cs_n_filters", "cs_frac_zlib_repro", "cs_n_hdr",
       "cs_n_zlevel", "cs_frac_sync", "cs_frac_adler", "cs_frac_junk", "cs_frac_len_indirect", "cs_ratio_std",
       "cs_ratio_max_z", "cs_n_sig", "cs_majority_share", "cs_n_singletons"]
L4 = ["upd_n_objs", "upd_touch_content", "upd_has_annot", "upd_has_page", "upd_has_sig"]
L5 = ["l5_n_annots", "l5_n_freetext", "l5_n_text_notes", "l5_n_img_xobj", "l5_n_fonts", "l5_n_std14_unembedded",
      "l5_frac_pages_multi_stream", "l5_max_streams_per_page"]
FULL = sorted(set(META + L1 + L23 + L4 + L5))
DOC = feat_df.loc[info.out_file.values, FULL].astype(float).values
dmu, dsd = np.nanmean(DOC[tr_idx], 0), np.nanstd(DOC[tr_idx], 0)
dsd[dsd < 1e-6] = 1.0
DOC = np.nan_to_num((DOC - dmu) / dsd, nan=0.0).astype(np.float32)

# ---------- PyG datasets (built once; raw arrays freed as we go) ----------
DATA = []
for i, g in enumerate(tqdm(graphs, desc="Building graphs")):
    XA = XS[i]
    ei = g["edge_index"].astype(np.int64)
    et = g["edge_type"].astype(np.int64)
    if ei.size:
        ei_t = torch.tensor(np.concatenate([ei, ei[::-1]], 1))
        et_t = torch.tensor(np.concatenate([et, et + NE]))
    else:
        ei_t, et_t = torch.zeros((2, 0), dtype=torch.long), torch.zeros(0, dtype=torch.long)
    npg = g["node_page"].astype(np.int64) if len(g["node_page"]) == XA.shape[0] else np.full(XA.shape[0], -1)
    DATA.append(Data(x=torch.tensor((XA - MU) / SD, dtype=torch.float32), edge_index=ei_t, edge_type=et_t,
                     y=torch.tensor([float(g["label"])]), node_page=torch.tensor(npg),
                     is_content=torch.tensor(XA[:, C_CONTENT] > 0.5),
                     doc=torch.tensor(DOC[i]).unsqueeze(0)))
    XS[i] = None
    g["edge_index"] = g["edge_type"] = None
del XS
gc.collect()
print("Node features after consistency:", len(ALL_COLS), "| document features:", DOC.shape[1])

# ---------- the model (identical to pilot CELL 14) ----------
R = 5.0
class Net(nn.Module):
    def __init__(self, cols, relational, content_pool, node_head, use_doc, hid=64, layers=3, drop=0.2):
        super().__init__()
        self.register_buffer("cols", torch.tensor(cols, dtype=torch.long))
        self.relational, self.content_pool, self.node_head, self.use_doc, self.drop = relational, content_pool, node_head, use_doc, drop
        self.inp = nn.Sequential(nn.Linear(len(cols), hid), nn.ReLU())
        if relational:
            self.etype = nn.Embedding(2 * NE, 16)
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for _ in range(layers):
            mlp = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, hid))
            self.convs.append(GINEConv(mlp, edge_dim=16) if relational else GINConv(mlp))
            self.norms.append(nn.LayerNorm(hid))
        d_cat = hid * (layers + 1)
        d_pool = d_cat * (4 if content_pool else 2) + (hid if use_doc else 0)
        if use_doc:
            self.doc_mlp = nn.Sequential(nn.Linear(DOC.shape[1], hid), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(d_pool, hid), nn.ReLU(), nn.Dropout(drop), nn.Linear(hid, 1))
        if node_head:
            self.node_mlp = nn.Sequential(nn.Linear(d_cat, hid), nn.ReLU(), nn.Linear(hid, 1))

    def forward(self, b):
        h = self.inp(b.x[:, self.cols])
        hs = [h]
        ea = self.etype(b.edge_type) if self.relational else None
        for conv, norm in zip(self.convs, self.norms):
            m = conv(h, b.edge_index, ea) if self.relational else conv(h, b.edge_index)
            h = h + F.dropout(F.relu(norm(m)), self.drop, self.training)
            hs.append(h)
        H = torch.cat(hs, 1)
        B = b.num_graphs
        parts = [scatter(H, b.batch, 0, dim_size=B, reduce="mean"), scatter(H, b.batch, 0, dim_size=B, reduce="max")]
        cm = b.is_content
        if self.content_pool:
            parts += [scatter(H[cm], b.batch[cm], 0, dim_size=B, reduce="mean"),
                      scatter(H[cm], b.batch[cm], 0, dim_size=B, reduce="max")]
        gvec = torch.nan_to_num(torch.cat(parts, 1), nan=0.0, posinf=0.0, neginf=0.0)
        if self.use_doc:
            gvec = torch.cat([gvec, self.doc_mlp(b.doc)], 1)
        logit = self.head(gvec).squeeze(-1)
        z = None
        if self.node_head:
            z = self.node_mlp(H).squeeze(-1)
            zc, bc = z[cm], b.batch[cm]
            cnt = scatter(torch.ones_like(zc), bc, 0, dim_size=B, reduce="sum")
            mx = torch.nan_to_num(scatter(zc, bc, 0, dim_size=B, reduce="max"), nan=0.0, posinf=0.0, neginf=0.0)
            s = scatter(torch.exp(R * (zc - mx[bc])), bc, 0, dim_size=B, reduce="mean")
            lse = mx + torch.log(s + 1e-12) / R
            logit = logit + torch.where(cnt > 0, lse, torch.zeros_like(lse))
        return logit, z

def make_model(kind):
    if kind == "GIN":
        return Net(BASE_COLS, relational=False, content_pool=False, node_head=False, use_doc=False)
    if kind == "CGNN":
        return Net(ALL_COLS, relational=True, content_pool=True, node_head=True, use_doc=False)
    if kind == "CGNN_DOC":
        return Net(ALL_COLS, relational=True, content_pool=True, node_head=True, use_doc=True)
    raise ValueError(kind)

@torch.no_grad()
def predict(model, items, want_pages=False):
    model.eval()
    S, P = [], []
    for b in DataLoader(items, batch_size=64, shuffle=False):
        out, z = model(b)
        S += torch.sigmoid(out).tolist()
        if want_pages:
            ptr = b.ptr.tolist()
            for a, c in zip(ptr[:-1], ptr[1:]):
                if z is None:
                    P.append(-1)
                    continue
                npg, zz = b.node_page[a:c], z[a:c]
                m = npg >= 0
                P.append(int(npg[m][torch.argmax(zz[m])]) if bool(m.any()) else -1)
    return np.array(S), P

def train_run(kind, seed, max_epochs=60, patience=10, lr=2e-3, bs=32):
    path = f"{RES_DIR}/{kind}_seed{seed}.csv"
    if os.path.exists(path):
        print(f"{kind} seed {seed}: already done, skipping ({path})")
        return
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = make_model(kind)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
    loader = DataLoader([DATA[i] for i in tr_idx], batch_size=bs, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    va_items = [DATA[i] for i in va_idx]
    yva = info.label.values[va_idx]
    best, best_state, best_ep, bad, log = -1.0, None, 0, 0, []
    t0 = time.time()
    for ep in range(1, max_epochs + 1):
        model.train()
        tot = 0.0
        for b in tqdm(loader, desc=f"{kind} seed {seed} | epoch {ep}", leave=False, mininterval=30):
            opt.zero_grad()
            out, _ = model(b)
            loss = F.binary_cross_entropy_with_logits(out, b.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tot += loss.item() * b.num_graphs
        va, _ = predict(model, va_items)
        auc = roc_auc_score(yva, va)
        sched.step(auc)
        if auc > best + 1e-4:
            best, best_state, best_ep, bad = auc, deepcopy(model.state_dict()), ep, 0
        else:
            bad += 1
        log.append({"epoch": ep, "loss": tot / len(tr_idx), "val_auc": auc, "lr": opt.param_groups[0]["lr"]})
        print(f"{kind} seed {seed} | epoch {ep:2d} | loss {tot / len(tr_idx):.4f} | val AUC {auc:.4f} | "
              f"best {best:.4f}@{best_ep} | {(time.time() - t0) / 60:.1f} min", flush=True)
        if bad >= patience:
            break
    model.load_state_dict(best_state)
    sc, pages = predict(model, [DATA[i] for i in te_idx], want_pages=True)
    out = info.loc[te_idx, ["out_file"]].copy()
    out["score"], out["pred_page"] = sc, pages
    out["kind"], out["seed"], out["best_val_auc"], out["best_epoch"] = kind, seed, best, best_ep
    pd.DataFrame(log).to_csv(f"{RES_DIR}/{kind}_seed{seed}_log.csv", index=False)
    out.to_csv(path, index=False)
    print(f"{kind} seed {seed}: best val AUC {best:.4f} at epoch {best_ep} | {(time.time() - t0) / 60:.1f} min -> saved {path}")

print(f"\nSetup complete in {(time.time() - t0) / 60:.1f} min. Next: CELL 25 (training).")

Graphs: /kaggle/input/notebooks/atharvagandhi/btp-full/full/full_graphs.pkl.gz
Loaded 40830 graphs in 8s | base node features: 43 | edge types: 12
Graphs -> train: 29685 | val: 5100 | dev: 6045
Docs   -> train: 1979 | val: 340 | dev: 403


Consistency features:   0%|          | 0/40830 [00:00<?, ?it/s]

Training nodes used for normalization: 13377555


Building graphs:   0%|          | 0/40830 [00:00<?, ?it/s]

Node features after consistency: 65 | document features: 61

Setup complete in 0.8 min. Next: CELL 25 (training).


In [1]:
# CELL 26 — Final dev comparison: GIN vs CGNN vs CGNN_DOC vs LightGBM (seeds, ensembles, twins, families, localization incl. combined rule+GNN, paired bootstrap)
import os, glob, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

hf = sorted(glob.glob("/kaggle/input/**/full/full_features.csv", recursive=True))
gfiles = sorted(glob.glob("/kaggle/input/**/gnn_full/*_seed[0-9].csv", recursive=True))
assert hf, "full_features.csv not found - is BTP Full attached?"
assert gfiles, "GNN result files not found - are the 3 GNN notebooks attached?"
OUT_DIR = "/kaggle/working/results"
os.makedirs(OUT_DIR, exist_ok=True)

res = pd.concat([pd.read_csv(f) for f in gfiles], ignore_index=True)
print("GNN result files:", len(gfiles), "| runs per model:", res.groupby("kind").seed.nunique().to_dict())

df = pd.read_csv(hf[0])
tr = df[df.split == "train"].reset_index(drop=True)
te = df[df.split == "dev"].reset_index(drop=True)
y = te.label.values
NO_EV = ["F2_digit_recompress", "F7_stamp_fake", "F8_textbox_fake", "G2_recompress", "G5_stamp_honest", "G6_textbox_honest"]
EV = ~te.variant.isin(NO_EV).values
print("Dev documents:", te.orig_file.nunique(), "| dev files:", len(te), "| evidence-bearing files:", int(EV.sum()))

# ---------- LightGBM reference (same features/params as CELL 23) ----------
META = ["has_producer", "producer_len", "moddate_eq_creation", "has_xmp", "tool_word_in_file"]
L1 = ["size_log", "version", "binary_comment", "n_eof", "n_startxref", "xref_table", "xref_stream", "obj_streams",
      "linearized", "true_revisions", "eol_crlf", "eol_cr", "eol_lf", "n_obj_defs", "obj_order_increasing",
      "obj_gen_nonzero", "obj_max_over_count", "obj_redefined", "obj_newline_dict", "dict_open_space",
      "dict_close_space", "stream_kw_crlf", "endobj_newline_before", "reals_dot_zero", "has_ID", "tool_word_in_file", "n_pages"]
L23 = ["cs_n", "cs_frac_flate", "cs_frac_lzw", "cs_frac_none", "cs_n_filters", "cs_frac_zlib_repro", "cs_n_hdr",
       "cs_n_zlevel", "cs_frac_sync", "cs_frac_adler", "cs_frac_junk", "cs_frac_len_indirect", "cs_ratio_std",
       "cs_ratio_max_z", "cs_n_sig", "cs_majority_share", "cs_n_singletons"]
L4 = ["upd_n_objs", "upd_touch_content", "upd_has_annot", "upd_has_page", "upd_has_sig"]
L5 = ["l5_n_annots", "l5_n_freetext", "l5_n_text_notes", "l5_n_img_xobj", "l5_n_fonts", "l5_n_std14_unembedded",
      "l5_frac_pages_multi_stream", "l5_max_streams_per_page"]
FULL = sorted(set(META + L1 + L23 + L4 + L5))
PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=10, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, random_state=42, verbose=-1)
lgb_scores = lgb.LGBMClassifier(**PARAMS).fit(tr[FULL], tr.label).predict_proba(te[FULL])[:, 1]

# ---------- helpers ----------
def tpr_at(yy, s, fpr):
    thr = np.quantile(s[yy == 0], 1 - fpr)
    return float((s[yy == 1] > thr).mean())

def auc(yy, s):
    try:
        return roc_auc_score(yy, s)
    except ValueError:
        return np.nan

def metrics(s):
    return {"AUC all": auc(y, s), "AUC evidence": auc(y[EV], s[EV]),
            "Det@1%FA evid": tpr_at(y[EV], s[EV], 0.01), "Det@5%FA evid": tpr_at(y[EV], s[EV], 0.05)}

def col_for(kind, seed, col="score"):
    r = res[res.kind == kind] if seed is None else res[(res.kind == kind) & (res.seed == seed)]
    v = r.groupby("out_file")[col].mean().reindex(te.out_file.values)
    assert not v.isna().any(), f"{kind} seed {seed}: missing dev predictions"
    return v.values

kinds = [k for k in ["GIN", "CGNN", "CGNN_DOC"] if k in res.kind.unique()]
seeds = {k: sorted(res[res.kind == k].seed.unique()) for k in kinds}

# ---------- 1. per-seed mean ± std ----------
print("\n=== 1. PER-SEED RESULTS: mean ± std over seeds (dev set) ===")
rows = []
for k in kinds:
    ms = pd.DataFrame([metrics(col_for(k, s)) for s in seeds[k]])
    rows.append({"Model": k, **{c: f"{ms[c].mean():.3f} ± {ms[c].std(ddof=0):.3f}" for c in ms.columns}})
rows.append({"Model": "LightGBM", **{c: f"{v:.3f}" for c, v in metrics(lgb_scores).items()}})
print(pd.DataFrame(rows).to_string(index=False))

# ---------- 2. seed ensembles ----------
ens = {f"{k} (3-seed ensemble)": col_for(k, None) for k in kinds}
ens["LightGBM"] = lgb_scores
print("\n=== 2. SEED-ENSEMBLE RESULTS (average of the seeds' scores) ===")
print(pd.DataFrame([{"Model": n, **{c: round(v, 3) for c, v in metrics(s).items()}} for n, s in ens.items()]).to_string(index=False))

# ---------- 3. twin pairs ----------
PAIRS = [("G1_resave", "F1_digit_resave"), ("G2_recompress", "F2_digit_recompress"), ("G3_note_incr", "F3_digit_incr"),
         ("G1_resave", "F4_value_resave"), ("G1_resave", "F5_cover_replace"), ("G4_linearize", "F6_digit_linearize"),
         ("G5_stamp_honest", "F7_stamp_fake"), ("G6_textbox_honest", "F8_textbox_fake")]
print("\n=== 3. TWIN-PAIR AUC (each forgery vs its honest twin) ===")
pr = []
for n, s in ens.items():
    r = {"Model": n}
    for g, f in PAIRS:
        m = te.variant.isin([g, f]).values
        r[f.split("_")[0]] = round(auc(y[m], s[m]), 3)
    pr.append(r)
print(pd.DataFrame(pr).to_string(index=False))

# ---------- 4. per family ----------
fams = sorted(te.family.unique())
print("\n=== 4. PER FAMILY, evidence-bearing AUC ===")
print("Dev documents per family:", {f.replace("acrobat ", "").replace("adobe ", ""): te[te.family == f].orig_file.nunique() for f in fams})
fr = []
for n, s in ens.items():
    r = {"Model": n}
    for fam in fams:
        m = (te.family == fam).values & EV
        r[fam.replace("acrobat ", "").replace("adobe ", "")] = round(auc(y[m], s[m]), 3)
    fr.append(r)
print(pd.DataFrame(fr).to_string(index=False))

# ---------- 5. localization: rule, GNN, and combined (rule when it fires, GNN otherwise) ----------
LOC_V = ["F1_digit_resave", "F4_value_resave", "F5_cover_replace", "F6_digit_linearize", "F2_digit_recompress"]
print("\n=== 5. PAGE LOCALIZATION on dev forgeries (% correct page) ===")
loc_rows = []
for v in LOC_V:
    m = (te.variant == v).values & te.edit_page.notna().values
    ep = te.edit_page.values[m]
    rule = te.loc_page.fillna(-1).values[m]
    fired = rule >= 0
    row = {"Forgery": v, "n": int(m.sum()),
           "Rule fires %": round(fired.mean() * 100, 1),
           "Rule correct when fires %": round((rule[fired] == ep[fired]).mean() * 100, 1) if fired.any() else np.nan}
    for k in [k for k in kinds if k != "GIN"]:
        g_acc, c_acc = [], []
        for s in seeds[k]:
            gp = col_for(k, s, "pred_page")[m]
            g_acc.append((gp == ep).mean() * 100)
            comb = np.where(fired, rule, gp)
            c_acc.append((comb == ep).mean() * 100)
        row[f"{k} alone %"] = f"{np.mean(g_acc):.1f} ± {np.std(g_acc):.1f}"
        row[f"Rule+{k} %"] = f"{np.mean(c_acc):.1f} ± {np.std(c_acc):.1f}"
    loc_rows.append(row)
print(pd.DataFrame(loc_rows).to_string(index=False))

# ---------- 6. paired bootstrap: each GNN ensemble minus LightGBM (evidence-bearing AUC) ----------
print("\n=== 6. PAIRED BOOTSTRAP vs LightGBM (evidence-bearing AUC, 1000 resamples over dev documents) ===")
rng = np.random.default_rng(0)
docs = te.orig_file.unique()
by_doc = {d: np.where((te.orig_file == d).values & EV)[0] for d in docs}
draws = [np.concatenate([by_doc[d] for d in rng.choice(docs, size=len(docs), replace=True)]) for _ in range(1000)]
for n in [n for n in ens if n != "LightGBM"]:
    diffs = np.array([auc(y[i], ens[n][i]) - auc(y[i], lgb_scores[i]) for i in tqdm(draws, desc=n, leave=False)])
    print(f"{n:28s}: mean diff {diffs.mean():+.4f} | 95% CI [{np.percentile(diffs, 2.5):+.4f}, {np.percentile(diffs, 97.5):+.4f}]"
          f" | GNN better in {(diffs > 0).mean() * 100:.1f}% of resamples")

# ---------- save ----------
out = te[["out_file", "orig_file", "family", "variant", "label", "tool", "edit_page", "loc_page"]].copy()
out["lightgbm"] = lgb_scores
for k in kinds:
    out[f"{k}_ens"] = ens[f"{k} (3-seed ensemble)"]
    for s in seeds[k]:
        out[f"{k}_s{s}_page"] = col_for(k, s, "pred_page")
out.to_csv(f"{OUT_DIR}/final_dev_comparison.csv", index=False)
print(f"\nSaved: {OUT_DIR}/final_dev_comparison.csv")

GNN result files: 9 | runs per model: {'CGNN': 3, 'CGNN_DOC': 3, 'GIN': 3}
Dev documents: 403 | dev files: 6045 | evidence-bearing files: 3627

=== 1. PER-SEED RESULTS: mean ± std over seeds (dev set) ===
   Model       AUC all  AUC evidence Det@1%FA evid Det@5%FA evid
     GIN 0.903 ± 0.001 0.997 ± 0.000 0.940 ± 0.006 0.958 ± 0.016
    CGNN 0.903 ± 0.000 0.997 ± 0.000 0.939 ± 0.006 0.986 ± 0.002
CGNN_DOC 0.907 ± 0.001 0.998 ± 0.000 0.959 ± 0.003 0.987 ± 0.016
LightGBM         0.908         0.998         0.965         0.999

=== 2. SEED-ENSEMBLE RESULTS (average of the seeds' scores) ===
                     Model  AUC all  AUC evidence  Det@1%FA evid  Det@5%FA evid
     GIN (3-seed ensemble)    0.904         0.998          0.952          0.987
    CGNN (3-seed ensemble)    0.904         0.997          0.951          0.986
CGNN_DOC (3-seed ensemble)    0.907         0.998          0.960          0.998
                  LightGBM    0.908         0.998          0.965          0.999

=== 

GIN (3-seed ensemble):   0%|          | 0/1000 [00:00<?, ?it/s]

GIN (3-seed ensemble)       : mean diff -0.0003 | 95% CI [-0.0012, +0.0008] | GNN better in 26.9% of resamples


CGNN (3-seed ensemble):   0%|          | 0/1000 [00:00<?, ?it/s]

CGNN (3-seed ensemble)      : mean diff -0.0011 | 95% CI [-0.0029, +0.0002] | GNN better in 5.3% of resamples


CGNN_DOC (3-seed ensemble):   0%|          | 0/1000 [00:00<?, ?it/s]

CGNN_DOC (3-seed ensemble)  : mean diff +0.0002 | 95% CI [-0.0007, +0.0013] | GNN better in 56.6% of resamples

Saved: /kaggle/working/results/final_dev_comparison.csv
